# Predicción de Dirección Financiera de TSLA mediante Redes Neuronales
---


## **Codigo de preprocesamiento y analisis exploratorio**
---

## Librerias




In [1]:
#Librerias Fase1
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import yfinance as yf


In [2]:
#Librerias Fase2
from sklearn.preprocessing import MinMaxScaler

In [ ]:
#Librerias Fase3
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.model_selection import train_test_split

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import Adam

In [ ]:
#Constantes
SEQUENCE_LENGTH = 30

In [ ]:
df = yf.download(
    "TSLA",
    start="2015-01-01",
    end="2025-12-31"
)

display(df.head())

In [ ]:
#Graficas
def plotTimeSeries(dataframe, column, title, plotType='line', startDate=None, endDate=None, xLabel='Fecha', yLabel='Valor', figureSize=(14, 6)):

    filteredData = dataframe.copy()

    if startDate is not None:
        filteredData = filteredData[filteredData.index >= startDate]

    if endDate is not None:
        filteredData = filteredData[filteredData.index <= endDate]

    seriesData = filteredData[column].squeeze()

    plt.figure(figsize=figureSize)

    if plotType == 'line':
        plt.plot(filteredData.index, seriesData)

    elif plotType == 'bar':
        plt.bar(filteredData.index, seriesData)

    plt.title(title)
    plt.xlabel(xLabel)
    plt.ylabel(yLabel)
    plt.grid(True)

    plt.show()

In [ ]:
def createTimeSeries(caracteristicasData, targetData, sequenceLength=30):

    X = []
    y = []

    for i in range(sequenceLength, len(caracteristicasData)):

        X.append(caracteristicasData[i-sequenceLength:i])
        y.append(targetData[i])

    return np.array(X), np.array(y)

In [ ]:
def createTimeSeriesBatches(caracteristicasData, targetData, sequenceLength=30, batchSize=32):

    X, y = createTimeSeries(caracteristicasData=caracteristicasData, targetData=targetData, sequenceLength=sequenceLength)

    XBatches = []
    yBatches = []

    for i in range(0, len(X), batchSize):

        XBatches.append(X[i:i+batchSize])
        yBatches.append(y[i:i+batchSize])

    return XBatches, yBatches

## Verificacion de valores faltantes y/o repetidos

In [ ]:
print("\nValores faltantes:")
print(df.isnull().sum())
print("\nDuplicados:")
print(df.duplicated().sum())

df.drop_duplicates(inplace=True)
df.dropna(inplace=True)

## Graficas de series de precio, volumen

In [ ]:
plotTimeSeries( dataframe=df,column='Close',title='Precio de Cierre TSLA',plotType='line',yLabel='Precio USD')
plotTimeSeries(dataframe=df,column='Volume',title='Volumen TSLA',plotType='bar',yLabel='Volumen')
plotTimeSeries(dataframe=df,column='Close',title='TSLA 2020-2022',plotType='line',startDate='2020-01-01',endDate='2022-12-31',yLabel='Precio USD')



In [ ]:
plotTimeSeries(dataframe=df,column='Close',title='Volumen TSLA (2018-2020)',startDate=None,endDate=None,yLabel='Volumen')

## Estadisticas descriptivas

In [ ]:
print("\nEstadísticas descriptivas:")
print(df.describe())

In [ ]:
print('Valor inicial df:',df.index.min())
print('valor final df  :',df.index.max())
print('Columnas df     :',df.columns)

## Preprocesamiento para generar series de tiempo etiquetas accion sube/ accion baja

In [ ]:
caracteristicas  = [ 'Open','High','Low','Close','Volume']

data = df[caracteristicas].copy()


data['SubeBaja'] = np.where(data['Close'].shift(-1) > data['Close'],1,0)
data.dropna(inplace=True)
print(data)

In [ ]:
scaler = MinMaxScaler(feature_range=(0,1))

caracteristicasEscaladas = scaler.fit_transform(data[caracteristicas])

In [ ]:

X, y = createTimeSeries(
    caracteristicasData=caracteristicasEscaladas,
    targetData=data['SubeBaja'].values,
    sequenceLength=SEQUENCE_LENGTH
)

print(X.shape)
print(y.shape)

In [ ]:
# print(X[0])


In [ ]:
XBatches, yBatches = createTimeSeriesBatches(caracteristicasData=caracteristicasEscaladas, targetData=data['SubeBaja'].values, sequenceLength=SEQUENCE_LENGTH , batchSize=32)

print(len(XBatches))
print(XBatches[0].shape)
print(yBatches[0].shape)

In [ ]:
#Graficaas
def plotLoss(history):

    plt.figure(figsize=(10,5))
    plt.plot(history.history['loss'], label='Train Loss')
    plt.plot(history.history['val_loss'], label='Validation Loss')
    plt.title("Training Loss vs Validation Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.grid(True)
    plt.show()


def plotPredictionVsReal(yReal, yPred):
    plt.figure(figsize=(12,6))
    plt.plot(yReal[:100], label='Real')
    plt.plot(yPred[:100], label='Predicción')
    plt.title("Prediction vs Real")
    plt.xlabel("Índice de muestra")
    plt.ylabel("Probabilidad")
    plt.legend()
    plt.grid(True)
    plt.show()

# Distribucion de clases

In [ ]:
plotBatchTargetDistribution(yBatches[0])

In [ ]:
def plotHistogram(data, title):

    plt.figure(figsize=(8,5))
    plt.hist(data, bins=30)
    plt.title(title)
    plt.grid(True)
    plt.show()

## Histograma de distribucion de precio cierre y volumenes

In [ ]:
plotHistogram(data['Close'],'Distribución Close')
plotHistogram(data['Volume'],'Distribución Volume')

Como era de esperarse por ser series financieras de acciones los datos No son gauseanos y tiene cola pesadas y volatilidad agrupada.

In [ ]:
def plotDensity(data, title, figureSize=(8,5)):
    seriesData = np.array(data).squeeze()
    pd.Series(seriesData).plot(kind='density',figsize=figureSize)
    plt.title(title)
    plt.grid(True)
    plt.show()

## Grafica de densidad segun precio

In [ ]:
plotDensity(data['Close'],'Densidad Close')

Valores posibles del precio X y la densidad relativa de esos valores en y. Entonces aca podemos visualizar donde se concentran los precios.

## Division de dataset con skelarn

In [ ]:
XTrain, XTest, yTrain, yTest = train_test_split(X,y,test_size=0.2,shuffle=False)

print(XTrain.shape)
print(XTest.shape)

In [ ]:
print("Train:", XTrain.shape)
print("Validation:", XValidation.shape)
print("Test:", XTest.shape)

## Definicion del modelo

In [ ]:
model = Sequential([
    LSTM(64,input_shape=(XTrain.shape[1],XTrain.shape[2])),
    Dense(1,activation='sigmoid')
])

## Entrenamiento del modelo

In [ ]:
model.compile(optimizer=Adam(learning_rate=0.0001), loss='binary_crossentropy', metrics=['accuracy'])


In [ ]:
model.summary()

In [ ]:
history = model.fit(XTrain, yTrain, epochs=30, batch_size=32, validation_split=0.2)


In [ ]:
plotLoss(history)


#Evaluacion del odelo

In [ ]:
yPred = model.predict(XTest)
yPredClass = (yPred > 0.5).astype(int)
accuracy = accuracy_score(yTest, yPredClass)
print("Accuracy:", accuracy)
print(confusion_matrix(yTest, yPredClass))

In [ ]:
plotPredictionVsReal(yTest, yPred)

In [ ]:
ultimaSecuencia = caracteristicasEscaladas[-sequenceLength:]

ultimaSecuencia = ultimaSecuencia.reshape(1, sequenceLength, XTrain.shape[2])

prediccionFutura = model.predict(ultimaSecuencia)

print("Predicción futura:")
print(prediccionFutura)